In [ ]:
import math


def genes_phen_plot(g):
    import numpy as np
    import pandas as pd
    import math
    from bokeh.models import BasicTicker, ColorBar, LinearColorMapper, ColumnDataSource, Legend
    from bokeh.plotting import figure
    from bokeh.transform import transform
    from bokeh.palettes import viridis
    
    # Taking the lists of genes and phenotypes
    gn = [i for i in g['MGI_Gene_ID']]
    ph = [j for i in g['Phenotype'] for j in i]

    # remove the duplicates from both the lists
    seen = set()
    seen_add = seen.add
    genes = [x for x in gn if not (x in seen or seen_add(x))]

    seen = set()
    seen_add = seen.add
    phen = [x for x in ph if not (x in seen or seen_add(x))]
    del (ph, gn, seen, seen_add)

    # Dictionary where the keys are genes and values the phenotypes related to them
    d = {}
    for i in range(0, len(genes)):
        d[genes[i]] = g['Phenotype'][g['MGI_Gene_ID'] == genes[i]].values[0]

    # Generating matrix where we have '1' if the phenotype is expressed by the gene (y axis phen, x axis genes)
    presence = np.zeros((len(phen), len(genes)))
    for i in range(len(phen)):
        for j in range(len(genes)):
            if phen[i] in d[genes[j]]:
                presence[i, j] = 1
    del (i, j)

    # formatting the data to use it with bokeh heatmap
    df = pd.DataFrame(presence, columns=genes, index=phen)
    df.index.name = 'Phenotypes'
    df.columns.name = 'Genes'
    df = df.stack().rename("value").reset_index()

    # colors used for the heatmap, you can use your own palette here (only 2 colors needed)
    colors = ['#1926d7', '#d71919']

    # Specific mapper to map color with value
    mapper = LinearColorMapper(palette=colors, low=df.value.min(), high=df.value.max())

    # Define a figure
    fig = figure(
                plot_width=1920,
                plot_height=1080,
                title="Genes to phenotypes relation",
                x_range=list(df.Genes.drop_duplicates()),
                y_range=list(df.Phenotypes.drop_duplicates()),
                toolbar_location='above',
                tools='pan,box_zoom,reset,save',
                x_axis_location="above",
                tooltips=[('Gene', '@Genes'), ('Phenotype', '@Phenotypes'), ('Relation', '@value')])

    fig.yaxis.major_label_orientation= math.pi/4
    fig.xaxis.major_label_orientation = math.pi / 4

    # Create rectangle for heatmap
    fig.rect(
                x="Genes",
                y="Phenotypes",
                width=1,
                height=1,
                source=ColumnDataSource(df),
                line_color="black",
                fill_color=transform('value', mapper))

    # Add legend
    color_bar = ColorBar(
                color_mapper=mapper,
                location=(0, 0),
                ticker=BasicTicker(desired_num_ticks=len(colors)),
                major_label_overrides={'1': '1 - Correlated', '0.5': '', '0': '0 - Not correlated'}
            )

    fig.add_layout(color_bar, 'right')
    return fig

def spiderchart_paths(p,p_list,phen):
    import io
    import numpy as np
    import pandas as pd
    from bokeh.io import output_file, show
    from bokeh.models import ColumnDataSource, LabelSet, Legend
    from bokeh.plotting import figure
    from bokeh.palettes import viridis
    from Bio.KEGG import REST
    import csv

    #Load the ENtrez id - MGI id mapping table
    ent_mgi = pd.read_table("modules/ENT_MGI.csv", header=0,index_col=[0])
    ent_mgi = ent_mgi.reset_index(drop=True)
    ent_mgi['ENTREZ_ID'] = ent_mgi['ENTREZ_ID'].astype('object')

    #Load KEGG pathway modules table and convert it in a dictonary
    KEGG_modules_mapping = pd.read_table("modules/KEGG_modules.csv",header=0, index_col=[0], sep="\t")
    mod_cat={}
    for i in range(0,len(KEGG_modules_mapping)):
        mod_cat[KEGG_modules_mapping['Mod_ID'][i]] = KEGG_modules_mapping['Macro_ID'][i]

    #Load the main category mapping table
    mod_cat_ID = pd.read_table("modules/main_Modules.csv", header=0, index_col=[0], sep="\t")

    # Use the Biopython KEGG module to get the KEGG pathway module IDs related to mouse genes
    x = REST.kegg_link('module','mmu')
    y = x.read()
    y = io.StringIO(y)
    mod_genes = pd.read_csv(y, sep='\t', header=None, names=('ENTREZ_ID', 'Mod_ID'))
    del(x,y)

    #Convert entrez id to mgi id in mod_genes
    #First we need to remove the string "mmu:" in the KEGG_ID column
    mod_genes['ENTREZ_ID'] = mod_genes['ENTREZ_ID'].map(lambda x: x.lstrip('mmu:'))

    #mapping mod_genes (ISSUE: for some reason we need to convert the ENTREZ_ID columns into string type since the object type is not working for the merge)
    mod_genes['ENTREZ_ID'] = mod_genes['ENTREZ_ID'].astype(str)
    ent_mgi['ENTREZ_ID'] = ent_mgi['ENTREZ_ID'].astype(str)
    mapped = pd.merge(mod_genes,ent_mgi, on=['ENTREZ_ID'])
    mapped = mapped.drop(['ENTREZ_ID'], axis=1)

    # Group by genes ID (and convert it into a dictonary to speed up search), in this way we'll obtain a list of modules for each gene
    mapped = mapped.groupby('MGI_ID')['Mod_ID'].apply(list).to_frame().reset_index()
    m = mapped.set_index('MGI_ID').T.to_dict('list')

    for i, row in p.iterrows():
        p['Pheno_data'][i]['Mod_ID']=np.NaN
        for j in row['Pheno_data']['MGI_ID']:
            if j in m.keys():
                row['Pheno_data']['Mod_ID'][row['Pheno_data']['MGI_ID'][row['Pheno_data']['MGI_ID']==j].index[0]] = m[j][0]
                print("True for " + j + " in position " + str(i))

    df = pd.DataFrame(columns=['Pheno_ID','Genes_count','Norm_count'])
    df['Pheno_ID'] = phen
    for i in range(0,len(df)):
        df['Genes_count'][i] = [0,0,0,0,0,0,0,0,0,0]
        df['Norm_count'][i] = [0,0,0,0,0,0,0,0,0,0]


    big_list = pd.concat(p_list).dropna().reset_index(drop=True).drop_duplicates(subset="MGI_ID", keep="first",inplace=False,ignore_index=True)
    for i in range(0,len(df)):
        for j in range(0,len(big_list)):
            if df['Pheno_ID'][i] in big_list['Pheno_list'][j]:
                for l in big_list['Mod_ID'][j]:
                    try:
                        if mod_cat[l] != 'X' or mod_cat[l] != '?':
                            df['Genes_count'][i][int(mod_cat[l])-1] += 1
                    except KeyError:
                        continue

    max = np.max(list(df['Genes_count']))
    min = np.min(list(df['Genes_count']))
    for i in range(0,len(df)):
        for j in range(0,len(df['Norm_count'][i])):
            df['Norm_count'][i][j] = (df['Genes_count'][i][j] - min) / (max-min)

    #GENERATING THE SPIDERCHART

    n_vertex = len(mod_cat_ID)-2
    theta = np.linspace(0, 2 * np.pi, n_vertex, endpoint=False)

    # rotate theta such that the first axis is at the top
    theta += np.pi / 2

    centre = 1

    #Labels for the circles to indicate the counting
    lab = pd.DataFrame(columns=['x', 'y', 'names'])
    lab['x'] = [0.2, 0.4, 0.6, 0.8, 1]
    lab['y'] = [centre,centre,centre,centre, centre]
    interval = np.linspace(0, max, 6)
    for i in range(0,len(interval)):
        interval[i] = int(interval[i])
    lab['names'] = interval[1:6]
    del (interval)

    def unit_poly_verts(theta, centre):
        """Return vertices of polygon for subplot axes.
        This polygon is circumscribed by a unit circle centered at (centre, centre)
        """
        x0, y0, r = [centre] * 3
        verts = [(r * np.cos(t) + x0, r * np.sin(t) + y0) for t in theta]
        return verts

    def radar_patch(r, theta, centre):
        """ Returns the x and y coordinates corresponding to the magnitudes of
        each variable displayed in the radar plot
        """
        # offset from centre of circle
        offset = 0
        yt = (r * centre + offset) * np.sin(theta) + centre
        xt = (r * centre + offset) * np.cos(theta) + centre
        return xt, yt

    verts = unit_poly_verts(theta, centre)
    x = [v[0] for v in verts]
    y = [v[1] for v in verts]
    x.append(verts[0][0])
    y.append(verts[0][1])

    fig = figure(title="Phenotype/Pathway module - Radar plot",height = 1980,width = 1080,toolbar_location='above',tools='pan,box_zoom,reset,save')
    text = mod_cat_ID['Name'][0:10].append(pd.Series(mod_cat_ID['Name'][0]))
    source = ColumnDataSource({'x': x , 'y': y , 'text': text})

    fig.line(x="x", y="y", source=source)

    labels = LabelSet(x="x", y="y", text="text", source=source)
    fig.add_layout(labels)

    cols = viridis(len(df))

    for i in range(len(df)):
        xt, yt = radar_patch(np.asarray(df['Norm_count'][i]), theta, centre)
        fig.patch(x=xt, y=yt, fill_alpha=0.15, fill_color=cols[i], line_color=cols[i], legend_label=str(df['Pheno_ID'][i]))

    # circular axes and lables
    radii = [0,0.2,0.4,0.6,0.8]
    fig.circle(centre, centre, radius=radii, fill_color=None, line_color="gray", line_alpha=0.15)
    fig.text(centre, radii, [str(r) for r in reversed(lab['names'])], text_font_size="5pt", text_align="center", text_baseline="middle", text_alpha=0.2)

    fig.axis.visible = False
    fig.xgrid.grid_line_color = None
    fig.ygrid.grid_line_color = None
    legend = Legend(items=fig.legend.items)
    fig.legend.visible=False
    legend.click_policy = 'hide'
    legend.visible=True
    fig.add_layout(legend,"left")
    return fig

def bar_plot_phen(p,p_list,terms):
    import numpy as np
    import pandas as pd
    import math
    from bokeh.io import show
    from bokeh.models import ColumnDataSource, LinearColorMapper, ColorBar, Legend
    from bokeh.plotting import figure
    from bokeh.transform import factor_cmap
    from bokeh.palettes import Turbo256, viridis
    import csv

    #add a column with the count of phenotype and one with the count of genes for each pathway
    p['phen_count'] = pd.NA
    p['gene_count'] = pd.NA
    for i in range(0,len(p)):
        if len(p_list[i]['Pheno_list']) == 0:
            p['phen_count'][i] = 0
        else:
            p['phen_count'][i] = len(list(dict.fromkeys(p_list[i]['Pheno_list'].apply(pd.Series).stack().reset_index(drop=True).to_list())))
        p['gene_count'][i] = len(p_list[i]['MGI_ID'])

    d = {k: v for v, k in enumerate(p.iloc[:,1].to_list())}
    l2 = []
    tmp = p.iloc[:,1]
    for i in terms:
        if i in tmp.unique():
            l2.append(p.iloc[d[i[0]]])
    df = pd.DataFrame(l2).reset_index(drop=True)
    #factors used by bokeh for the count colors for genes
    fac = []
    for i in range(0,len(df)):
        fac.append(str(df['gene_count'][i]))
    fac = pd.Series(fac)
    #here  if the number of count it's bigger than the size of the color palette, we are going to
    #divide the counts and colors into ranges. Otherwise we are going to use the default palette
    if len(fac.unique()) > 256:
        step = df['gene_count'].max() / 256
        bounds = [round(step * i) for i in range(256)]
        bound_colors = []
        j = 0
        x = False
        for i in range(0, df['gene_count'].max()):
            if not x:
                if i >= bounds[j + 1]:
                    j += 1
                    if j+1 == 255:
                        x=True
                bound_colors.append(Turbo256[j])
            else:
                bound_colors.append(Turbo256[255])
        source = ColumnDataSource(data=dict(path=df.iloc[:,1], counts=df['phen_count'], g_counts=fac))
        my_cmap = LinearColorMapper(palette=bound_colors, low=0, high=df['gene_count'].max())


        fig = figure(title="Phenotype/genes count per pathway",width=1280, height=720, x_range=df.iloc[:,1], tooltips=[('Gene count', '@g_counts'), ('Phenotype count', '@counts')],toolbar_location='above',tools='pan,box_zoom,reset,save')
        fig.vbar(x='path', top='counts', width=0.9, source=source, line_color='white', fill_color={"field":"g_counts","transform":my_cmap})
        fig.xgrid.grid_line_color = None
        fig.y_range.start = 0
        fig.y_range.end = np.max(df['phen_count'].values) + 100
        fig.xaxis.major_label_orientation = math.pi / 2
    else:
        source = ColumnDataSource(data=dict(path=df.iloc[:,1], counts=df['phen_count'], g_counts=fac))
        colors = factor_cmap('g_counts', palette=viridis(len(fac.unique())), factors=fac.unique())
        fig = figure(title="Phenotype/genes count per pathway", width=1280, height=720, x_range=df.iloc[:,1], tooltips=[('Gene count', '@g_counts'), ('Phenotype count', '@counts')],toolbar_location='above',tools='pan,box_zoom,reset,save')
        fig.vbar(x='path', top='counts', width=0.9, source=source, line_color='white', fill_color=colors)
        fig.xgrid.grid_line_color = None
        fig.y_range.start = 0
        fig.y_range.end = np.max(df['phen_count'].values) + 100
        fig.xaxis.major_label_orientation = math.pi / 2
    
        
    return fig